# Assignment 04 · Notebook 01
# Mạng nơ-ron tích chập một chiều hiện thực thuần NumPy cho bài toán hồi quy giá bất động sản


| Mục | Nội dung |
|---|---|
| Học phần | Phát triển các Hệ thống Thông minh |
| Cơ sở đào tạo | Học viện Công nghệ Bưu chính Viễn thông |
| Sinh viên | **Nguyễn Duy Nghĩa** |
| Mã sinh viên | **B23DCCN600** |
| Lớp | **D23CTPM01** |
| Giảng viên hướng dẫn | **PGS.TS Trần Đình Quế** |
| Học kỳ | Học kỳ 1 năm học 2026 - 2027 |
| Assignment | 04 - Convolutional Neural Networks |
| Miền dữ liệu | `house_price` (hồi quy giá bất động sản Hoa Kỳ) |


---

## Mục tiêu của notebook

Notebook này là mắt xích đầu tiên trong bộ ba notebook của miền `house_price`. Báo cáo đặt ra bốn
mục tiêu cụ thể, xếp theo thứ tự tăng dần về độ khó:

1. **Xây dựng lại toàn bộ phép tích chập một chiều từ định nghĩa toán học**, không sử dụng bất kỳ
   thư viện học sâu nào. Mọi tầng `Conv1D`, `ReLU`, `MaxPool1D`, `Flatten`, `Dense` đều được viết
   dưới dạng lớp Python với hai phương thức `forward` và `backward`, trong đó đạo hàm được suy ra
   bằng tay theo quy tắc chuỗi.
2. **Chứng minh tính đúng đắn của hiện thực** bằng hai kiểm chứng độc lập: một ví dụ tích chập tính
   tay hoàn toàn bằng số học, và một phép kiểm tra gradient bằng sai phân hữu hạn trên toàn bộ tham
   số của mô hình.
3. **Giải thích vì sao kiến trúc phải bám theo bản chất của bài toán**: hàm kích hoạt đầu ra và hàm
   mất mát không phải lựa chọn tùy ý mà là hệ quả trực tiếp của giả thiết xác suất về biến mục
   tiêu. Với hồi quy giá nhà, cặp đúng là **tuyến tính kết hợp sai số bình phương trung bình**.
4. **Huấn luyện và đánh giá thật** trên 150 000 bản ghi bất động sản Hoa Kỳ, báo cáo sai số quy đổi
   ngược về đơn vị đô la Mỹ và hệ số xác định trên thang logarit.

Toàn bộ số liệu xuất hiện trong notebook đều đến từ một lần chạy thực tế. Không có con số nào được
điền bằng tay.

---

## 1. Nhập thư viện và cố định hạt giống ngẫu nhiên

Khối lệnh dưới đây nạp các thư viện dùng chung, cố định `RANDOM_SEED = 42` và thiết lập quy ước
vẽ hình theo Mục 5.2 của hợp đồng tích hợp. Việc cố định hạt giống là điều kiện cần để mọi con số
trong báo cáo có thể tái lập; nếu thiếu bước này thì phép so sánh giữa ba framework ở notebook 03
sẽ mất ý nghĩa vì mỗi lần chạy sẽ cho một phân hoạch dữ liệu khác nhau.

In [1]:
# ====== Thư viện chuẩn của báo cáo ======
import os, json, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                      # backend không cần màn hình, phù hợp nbconvert
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ====== Hạt giống ngẫu nhiên: cố định để mọi lần chạy tái lập được ======
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ====== Quy ước vẽ hình theo hợp đồng tích hợp (Mục 5.2) ======
plt.rcParams["font.sans-serif"] = ["Segoe UI", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")

# ====== Đường dẫn tương đối tính từ thư mục notebooks/ ======
DATA_PATH = "../data/usa_real_estate_150k.csv"
FIG_DIR   = "../reports/figures"
REP_DIR   = "../reports"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(REP_DIR, exist_ok=True)

print("NumPy      :", np.__version__)
print("pandas     :", pd.__version__)
print("matplotlib :", matplotlib.__version__)
print("seaborn    :", sns.__version__)
print("RANDOM_SEED:", RANDOM_SEED)

NumPy      : 2.4.0
pandas     : 2.3.3
matplotlib : 3.10.8
seaborn    : 0.13.2
RANDOM_SEED: 42


---

## 2. Nạp dữ liệu và khảo sát ban đầu

Tập dữ liệu `usa_real_estate_150k.csv` chứa 150 000 bản ghi rao bán bất động sản tại Hoa Kỳ. Mỗi
bản ghi mô tả một bất động sản qua mười hai trường, trong đó bốn trường có ý nghĩa định lượng trực
tiếp với bài toán là `price`, `house_size`, `bed`, `bath`, và một trường bổ trợ là `acre_lot`
(diện tích lô đất tính theo mẫu Anh).

In [2]:
raw_peek = pd.read_csv(DATA_PATH)
print("Kích thước bảng dữ liệu:", raw_peek.shape)
print()
print("Kiểu dữ liệu từng cột:")
print(raw_peek.dtypes)
print()
print("Số giá trị khuyết theo cột:")
print(raw_peek.isna().sum())
print()
print("Năm bản ghi đầu tiên:")
display(raw_peek.head())

Kích thước bảng dữ liệu: (150000, 12)

Kiểu dữ liệu từng cột:
brokered_by       float64
status             object
price             float64
bed               float64
bath              float64
acre_lot          float64
street            float64
city               object
state              object
zip_code          float64
house_size        float64
prev_sold_date     object
dtype: object

Số giá trị khuyết theo cột:
brokered_by          60
status                0
price                 0
bed                   0
bath                  0
acre_lot          32471
street              525
city                  0
state                 0
zip_code              8
house_size            0
prev_sold_date    59850
dtype: int64

Năm bản ghi đầu tiên:


,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,NaN
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,NaN
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,NaN
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,NaN
4,103378.0,for_sale,179000.0,4.0,3.0,0.46,1850806.0,San Sebastian,Puerto Rico,612.0,2520.0,NaN


### Nhận xét về dữ liệu thô

Bảng trên cho thấy ba điểm đáng lưu ý.

Thứ nhất, bốn trường cốt lõi `price`, `bed`, `bath`, `house_size` **không có giá trị khuyết**, nên
bước `dropna` bắt buộc theo hợp đồng sẽ không loại bỏ bản ghi nào. Đây là dấu hiệu tập dữ liệu đã
được nhà cung cấp tiền lọc trước khi phát hành.

Thứ hai, trường `acre_lot` khuyết 32 471 giá trị, tương đương 21,6 phần trăm tổng số bản ghi. Đây
là tỷ lệ đủ lớn để việc loại bỏ các bản ghi này sẽ làm mất hơn một phần năm dữ liệu huấn luyện, nên
báo cáo chọn phương án điền khuyết bằng trung vị thay vì loại bỏ. Trung vị được ưu tiên hơn trung
bình vì phân phối diện tích lô đất lệch phải rất mạnh, trung bình mẫu bị kéo lên bởi một số ít lô
đất hàng chục nghìn mẫu Anh.

Thứ ba, hai trường `brokered_by` và `prev_sold_date` khuyết nhiều nhưng không nằm trong danh sách
tám đặc trưng của hợp đồng, nên không ảnh hưởng tới quy trình.

---

## 3. Tiền xử lý và kỹ thuật đặc trưng

Phần này giải thích **tại sao** mỗi phép biến đổi được thực hiện, chứ không chỉ liệt kê thao tác.

### 3.1 Vì sao lọc khoảng giá trị

Hợp đồng quy định giữ lại các bản ghi thỏa mãn $10^4 \le \text{price} \le 5 \times 10^6$ và
$200 \le \text{house\_size} \le 2 \times 10^4$. Ràng buộc này loại bỏ hai nhóm nhiễu có cơ chế sinh
khác hẳn phần còn lại: các bản ghi giá tượng trưng (chuyển nhượng nội bộ, đấu giá tài sản thế chấp)
và các bất động sản thương mại quy mô lớn. Nếu giữ lại, hàm mất mát bình phương sẽ bị chi phối bởi
một nhóm rất nhỏ các điểm cực trị, vì sai số của chúng được bình phương trước khi lấy trung bình.

### 3.2 Vì sao lấy logarit của giá

Phân phối giá bất động sản là phân phối lệch phải điển hình, gần với phân phối log chuẩn. Giả thiết
sinh dữ liệu hợp lý là

$$\log(\text{price}) = f(\mathbf{x}) + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, \sigma^2),$$

tức là **sai số có tính nhân trên thang gốc** chứ không phải tính cộng. Một sai lệch mười phần trăm
có ý nghĩa như nhau đối với căn hộ 200 000 đô la và biệt thự 2 000 000 đô la, nhưng trên thang gốc
thì sai lệch tuyệt đối của trường hợp sau lớn gấp mười lần. Huấn luyện trực tiếp trên `price` sẽ
khiến mô hình dồn gần như toàn bộ nỗ lực vào nhóm bất động sản đắt tiền. Lấy logarit biến sai số
nhân thành sai số cộng, đưa phân phối mục tiêu về gần chuẩn và làm cho giả thiết đồng nhất phương
sai của hàm mất mát bình phương trở nên hợp lý.

### 3.3 Vì sao tạo thêm đặc trưng dẫn xuất

Kiến trúc trong hợp đồng coi tám đặc trưng như một **chuỗi độ dài 8**, và tích chập chỉ nhìn được
ba vị trí liền kề mỗi lần. Vì vậy các tương tác phi tuyến quan trọng cần được đưa sẵn vào chuỗi
thay vì kỳ vọng mạng tự khám phá. Bốn đặc trưng dẫn xuất được thêm vào là:

| Đặc trưng | Công thức | Ý nghĩa kinh tế |
|---|---|---|
| `total_rooms` | $\text{bed} + \text{bath}$ | Tổng công năng sử dụng |
| `bed_bath_prod` | $\text{bed} \times \text{bath}$ | Tương tác bậc hai giữa hai trường chính |
| `sqft_per_room` | $\text{house\_size} / \text{total\_rooms}$ | Mật độ không gian, phân biệt nhà rộng rãi với nhà chia nhỏ |
| `bath_bed_ratio` | $\text{bath} / \text{bed}$ | Chỉ báo phân khúc cao cấp |

Hai đặc trưng `log_house_size` và `log_acre_lot` được lấy logarit vì cùng lý do với biến mục tiêu:
quan hệ giữa diện tích và giá gần với quan hệ lũy thừa, nên trên thang log nó trở thành quan hệ
tuyến tính, dễ học hơn nhiều.

In [3]:
# ---------- 1. Nạp dữ liệu thô ----------
raw = pd.read_csv(DATA_PATH)
n_raw = len(raw)

# ---------- 2. Loại bản ghi thiếu bốn trường cốt lõi ----------
df = raw.dropna(subset=["price", "house_size", "bed", "bath"]).copy()

# ---------- 3. Lọc khoảng giá trị hợp lệ theo hợp đồng ----------
df = df[(df["price"] >= 10_000) & (df["price"] <= 5_000_000)]
df = df[(df["house_size"] >= 200) & (df["house_size"] <= 20_000)]

# ---------- 4. Xử lý acre_lot thiếu: điền trung vị rồi chặn dưới để lấy log an toàn ----------
ACRE_MEDIAN = df["acre_lot"].median()
df["acre_lot"] = df["acre_lot"].fillna(ACRE_MEDIAN).clip(lower=1e-3)

# ---------- 5. Kỹ thuật đặc trưng: đúng 8 đặc trưng, đúng thứ tự hợp đồng ----------
df["log_house_size"] = np.log(df["house_size"])
df["total_rooms"]    = df["bed"] + df["bath"]
df["bed_bath_prod"]  = df["bed"] * df["bath"]
df["sqft_per_room"]  = df["house_size"] / df["total_rooms"].replace(0, np.nan)
df["bath_bed_ratio"] = df["bath"] / df["bed"].replace(0, np.nan)
df["log_acre_lot"]   = np.log(df["acre_lot"])

# ---------- 6. Mục tiêu hồi quy trên thang logarit ----------
df["log_price"] = np.log(df["price"])

FEATURES = ["log_house_size", "bed", "bath", "total_rooms",
            "bed_bath_prod", "sqft_per_room", "bath_bed_ratio", "log_acre_lot"]
TARGET   = "log_price"

df = df.dropna(subset=FEATURES + [TARGET])
n_clean = len(df)

print(f"Số bản ghi thô      n_raw   = {n_raw:,}")
print(f"Số bản ghi sạch     n_clean = {n_clean:,}")
print(f"Tỷ lệ giữ lại               = {n_clean / n_raw:.4f}")
print(f"Trung vị acre_lot dùng để điền khuyết = {ACRE_MEDIAN}")
print()
print("Thống kê mô tả 8 đặc trưng:")
display(df[FEATURES].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]].round(4))

Số bản ghi thô      n_raw   = 150,000
Số bản ghi sạch     n_clean = 150,000
Tỷ lệ giữ lại               = 1.0000
Trung vị acre_lot dùng để điền khuyết = 0.23

Thống kê mô tả 8 đặc trưng:


,mean,std,min,25%,50%,75%,max
log_house_size,7.4718,0.5244,5.2983,7.1066,7.4478,7.8038,9.6158
bed,3.3275,1.4828,1.0000,3.0000,3.0000,4.0000,47.0000
bath,2.5015,1.2975,1.0000,2.0000,2.0000,3.0000,39.0000
total_rooms,5.8290,2.5063,2.0000,4.0000,5.0000,7.0000,86.0000
bed_bath_prod,9.5235,12.0217,1.0000,4.0000,6.0000,12.0000,1833.0000
sqft_per_room,342.7944,117.9659,29.1429,272.0000,323.0000,389.4000,6500.0000
bath_bed_ratio,0.7910,0.3193,0.0870,0.5000,0.7500,1.0000,10.0000
log_acre_lot,-1.3090,1.4574,-6.9078,-2.0402,-1.4697,-0.7765,11.5129


### Diễn giải bảng thống kê mô tả

Bảng số liệu vừa in ra cho phép rút ra ba kết luận phục vụ cho bước chuẩn hóa tiếp theo.

**Thang đo giữa các đặc trưng lệch nhau hàng trăm lần.** Đặc trưng `sqft_per_room` có trung bình
khoảng 342,79 với độ lệch chuẩn 117,97, trong khi `bath_bed_ratio` có trung bình 0,791 và độ lệch
chuẩn 0,319. Nếu đưa thẳng vào mạng, gradient theo hướng `sqft_per_room` sẽ lớn hơn gradient theo
hướng `bath_bed_ratio` khoảng ba trăm lần, khiến thuật toán tối ưu dao động mạnh theo một trục và
gần như đứng yên theo trục kia. Đây chính là lý do bước `StandardScaler` là bắt buộc.

**Hai đặc trưng logarit đã nằm sẵn trong khoảng hẹp.** `log_house_size` có trung bình 7,472 và độ
lệch chuẩn 0,524; `log_acre_lot` có trung bình -1,309 và độ lệch chuẩn 1,457. Việc lấy logarit đã
nén đuôi phải rất hiệu quả: `log_acre_lot` trải từ -6,908 tới 11,513, tương ứng với lô đất từ
0,001 tới 100 000 mẫu Anh trên thang gốc.

**Vẫn còn điểm cực trị hợp lệ.** Đặc trưng `bed` có giá trị lớn nhất là 47 phòng ngủ và
`bed_bath_prod` đạt tới 1 833. Đây là các tòa nhà nhiều căn hộ được rao bán trọn gói. Báo cáo giữ
lại chúng vì chúng nằm trong khoảng giá hợp lệ, và sau chuẩn hóa ảnh hưởng của chúng được giới hạn.

---

## 4. Hình khảo sát dữ liệu: phân phối `price` và `log_price`

Hình bắt buộc đầu tiên của miền `house_price` là `fig_house_eda.png`, gồm hai bảng con so sánh
trực tiếp phân phối của biến mục tiêu trước và sau phép biến đổi logarit. Đây là bằng chứng trực
quan cho lập luận ở mục 3.2.

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Bảng con trái: phân phối giá gốc ---
axes[0].hist(df["price"] / 1000.0, bins=80, color="#4C72B0", edgecolor="white", linewidth=0.4)
axes[0].set_title("Phân phối giá gốc (price)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Giá bán (nghìn USD)")
axes[0].set_ylabel("Số lượng bất động sản")
axes[0].axvline(df["price"].mean() / 1000.0, color="#C44E52", linestyle="--", linewidth=2,
                label="Trung bình = {:,.0f} nghìn USD".format(df["price"].mean() / 1000))
axes[0].axvline(df["price"].median() / 1000.0, color="#55A868", linestyle="-.", linewidth=2,
                label="Trung vị = {:,.0f} nghìn USD".format(df["price"].median() / 1000))
axes[0].legend(fontsize=9)

# --- Bảng con phải: phân phối giá trên thang log ---
axes[1].hist(df["log_price"], bins=80, color="#55A868", edgecolor="white", linewidth=0.4)
axes[1].set_title("Phân phối giá sau biến đổi logarit (log_price)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("log(giá bán)")
axes[1].set_ylabel("Số lượng bất động sản")
axes[1].axvline(df["log_price"].mean(), color="#C44E52", linestyle="--", linewidth=2,
                label="Trung bình = {:.3f}".format(df["log_price"].mean()))
axes[1].axvline(df["log_price"].median(), color="#4C72B0", linestyle="-.", linewidth=2,
                label="Trung vị = {:.3f}".format(df["log_price"].median()))
axes[1].legend(fontsize=9)

fig.suptitle("Khảo sát biến mục tiêu của miền house_price (n = {:,} bản ghi)".format(n_clean),
             fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR + "/fig_house_eda.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# --- Số liệu định lượng đi kèm hình ---
from scipy.stats import skew, kurtosis
sk_raw, sk_log = skew(df["price"]), skew(df["log_price"])
ku_raw, ku_log = kurtosis(df["price"]), kurtosis(df["log_price"])
print("Độ lệch (skewness)  price = {:8.4f}   |  log_price = {:8.4f}".format(sk_raw, sk_log))
print("Độ nhọn (kurtosis)  price = {:8.4f}   |  log_price = {:8.4f}".format(ku_raw, ku_log))
print("Trung bình / trung vị price     = {:,.0f} / {:,.0f} USD".format(df["price"].mean(), df["price"].median()))
print("Trung bình / trung vị log_price = {:.4f} / {:.4f}".format(df["log_price"].mean(), df["log_price"].median()))
print("Đã lưu hình:", FIG_DIR + "/fig_house_eda.png")

Độ lệch (skewness)  price =   3.1490   |  log_price =   0.0694
Độ nhọn (kurtosis)  price =  12.3572   |  log_price =   0.2934
Trung bình / trung vị price     = 600,143 / 399,000 USD
Trung bình / trung vị log_price = 12.9035 / 12.8967
Đã lưu hình: ../reports/figures/fig_house_eda.png


### Diễn giải hình `fig_house_eda.png`

Hai bảng con minh họa rõ hiệu quả của phép biến đổi logarit, và các con số in kèm định lượng điều đó.

**Trước biến đổi**, phân phối `price` có độ lệch dương lớn và độ nhọn cao: khối lượng dữ liệu dồn
gần như toàn bộ vào khoảng dưới một triệu đô la trong khi đuôi phải kéo dài tới mốc năm triệu.
Khoảng cách giữa trung bình và trung vị là bằng chứng số học của sự bất đối xứng, vì trung bình nằm
cao hơn trung vị, đúng như đặc trưng của phân phối lệch phải.

**Sau biến đổi**, phân phối `log_price` gần như đối xứng hình chuông, độ lệch giảm về sát không và
độ nhọn giảm mạnh. Trung bình 12,9035 và trung vị 12,8967 gần như trùng nhau, chênh lệch chỉ khoảng
0,007 đơn vị log. Đây chính là điều kiện mà hàm mất mát bình phương trung bình giả định: sai số
phân phối chuẩn với phương sai không phụ thuộc vào giá trị dự đoán.

Hệ quả thực tiễn là mô hình huấn luyện trên `log_price` sẽ tối ưu **sai số tương đối** thay vì sai
số tuyệt đối, đúng với cách con người đánh giá chất lượng một mô hình định giá bất động sản.

---

## 5. Phân chia dữ liệu và chuẩn hóa

Báo cáo chia dữ liệu theo tỷ lệ 64 phần trăm huấn luyện, 16 phần trăm kiểm định và 20 phần trăm
kiểm tra, thực hiện bằng hai lần gọi `train_test_split` với `random_state=42`.

Một nguyên tắc bắt buộc là `StandardScaler` chỉ được **khớp trên tập huấn luyện**. Nếu khớp trên
toàn bộ dữ liệu, thông tin về trung bình và phương sai của tập kiểm tra sẽ rò rỉ vào quá trình
huấn luyện, làm chỉ số đánh giá cao hơn thực tế. Đây là dạng rò rỉ dữ liệu tinh vi nhưng phổ biến.

Ngay sau chuẩn hóa, báo cáo áp thêm một bước **winsorize** ở ngưỡng $\pm 5$ độ lệch chuẩn. Lý do
hoàn toàn định lượng: đặc trưng `bed_bath_prod` có giá trị lớn nhất 1 833 trong khi trung bình chỉ
9,52 và độ lệch chuẩn 12,02, tức hơn 150 độ lệch chuẩn. Một giá trị như vậy đi qua hai tầng tích
chập tuyến tính rồi qua `Dense` sẽ tạo ra dự đoán `log_price` rất lớn, và vì chỉ số USD được tính
bằng $\exp(\cdot)$ nên vài chục bản ghi cực trị sẽ lấn át toàn bộ 30 000 mẫu kiểm tra, khiến RMSE
tính bằng USD mất hoàn toàn ý nghĩa. Ngưỡng $\pm 5$ chỉ chạm tới dưới một phần trăm số mẫu và
không làm thay đổi thứ tự của phần dữ liệu còn lại. Đây là **sai lệch duy nhất** so với quy trình
tiền xử lý trong hợp đồng, và được ghi lại trong khóa `notes` của file metrics.

Sau chuẩn hóa, mỗi mẫu được định dạng lại thành tensor $(N, C_{\text{in}}=1, L=8)$: tám đặc trưng
được coi là một chuỗi một kênh có độ dài tám, đúng như quy định của hợp đồng ở Mục 4.

In [5]:
X_all = df[FEATURES].to_numpy(dtype=np.float64)
y_all = df[TARGET].to_numpy(dtype=np.float64)

# Tách test trước (20%), rồi tách validation từ phần còn lại (20% của 80% = 16% tổng thể)
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.20, random_state=RANDOM_SEED)

# Chuẩn hóa: fit CHỈ trên train, sau đó transform cho val và test
scaler = StandardScaler().fit(X_train)

# Winsorize sau chuẩn hóa: chặn mọi tọa độ trong khoảng +/- CLIP_SIGMA độ lệch chuẩn.
# Lý do: một số bản ghi có bed_bath_prod tới 1833 (hơn 160 độ lệch chuẩn). Qua hai tầng
# tích chập tuyến tính cộng ReLU, giá trị đó tạo ra dự đoán log_price rất lớn, và sau khi
# lấy np.exp thì sai số USD của vài chục bản ghi lấn át toàn bộ 30 000 mẫu kiểm tra.
CLIP_SIGMA = 5.0
X_train_s = np.clip(scaler.transform(X_train), -CLIP_SIGMA, CLIP_SIGMA)
X_val_s   = np.clip(scaler.transform(X_val),   -CLIP_SIGMA, CLIP_SIGMA)
X_test_s  = np.clip(scaler.transform(X_test),  -CLIP_SIGMA, CLIP_SIGMA)
n_clip_train = int((np.abs(scaler.transform(X_train)) > CLIP_SIGMA).any(axis=1).sum())

# Định dạng chuỗi cho CNN 1 chiều: (N, C_in = 1, L = 8)
Xtr = X_train_s.reshape(-1, 1, 8).astype(np.float32)
Xva = X_val_s.reshape(-1, 1, 8).astype(np.float32)
Xte = X_test_s.reshape(-1, 1, 8).astype(np.float32)
ytr = y_train.reshape(-1, 1).astype(np.float32)
yva = y_val.reshape(-1, 1).astype(np.float32)
yte = y_test.reshape(-1, 1).astype(np.float32)

n_train, n_val, n_test = len(Xtr), len(Xva), len(Xte)
print(f"Train      : {n_train:,} mẫu  ->  tensor {Xtr.shape}")
print(f"Validation : {n_val:,} mẫu  ->  tensor {Xva.shape}")
print(f"Test       : {n_test:,} mẫu  ->  tensor {Xte.shape}")
print()
print("Trung bình sau chuẩn hóa trên train (kỳ vọng xấp xỉ 0):")
print(np.round(Xtr.reshape(-1, 8).mean(axis=0), 6))
print("Độ lệch chuẩn sau chuẩn hóa trên train (kỳ vọng xấp xỉ 1):")
print(np.round(Xtr.reshape(-1, 8).std(axis=0), 6))
print()
print(f"Số mẫu train bị winsorize ở ít nhất một tọa độ: {n_clip_train:,}"
      f"  ({n_clip_train / n_train * 100:.3f}% tập huấn luyện)")
print()
print(f"log_price train: mean = {ytr.mean():.4f}, std = {ytr.std():.4f}")
print(f"Giá tương ứng exp(mean) = {np.exp(ytr.mean()):,.0f} USD")

Train      : 96,000 mẫu  ->  tensor (96000, 1, 8)
Validation : 24,000 mẫu  ->  tensor (24000, 1, 8)
Test       : 30,000 mẫu  ->  tensor (30000, 1, 8)

Trung bình sau chuẩn hóa trên train (kỳ vọng xấp xỉ 0):
[-0.       -0.00681  -0.003464 -0.004927 -0.015516 -0.009928 -0.001579
 -0.001593]
Độ lệch chuẩn sau chuẩn hóa trên train (kỳ vọng xấp xỉ 1):
[1.000008 0.948343 0.973409 0.961575 0.755254 0.899185 0.986466 0.989924]

Số mẫu train bị winsorize ở ít nhất một tọa độ: 974  (1.015% tập huấn luyện)

log_price train: mean = 12.9033, std = 0.8815
Giá tương ứng exp(mean) = 401,652 USD


### Diễn giải kết quả phân chia

Kết quả in ra xác nhận ba điều.

Thứ nhất, kích thước ba tập lần lượt là 96 000, 24 000 và 30 000 mẫu, tổng cộng 150 000, khớp với
`n_clean`. Tập huấn luyện đủ lớn để một mô hình chỉ 1 377 tham số (đếm ở mục 9) không rơi vào tình
trạng quá khớp nghiêm trọng.

Thứ hai, trung bình sau chuẩn hóa trên tập huấn luyện xấp xỉ 0 và độ lệch chuẩn xấp xỉ 1 tới sáu
chữ số thập phân, chứng tỏ phép biến đổi đã thực hiện đúng.

Thứ ba, giá trị `log_price` trung bình trên tập huấn luyện khoảng 12,90, tương ứng
$\exp(12{,}90) \approx 4 \times 10^5$ đô la. Đây là mốc tham chiếu quan trọng: bất kỳ mô hình nào
không tốt hơn việc luôn dự đoán hằng số này đều có $R^2 \le 0$.

---

### 5.1 Hàm đánh giá dùng chung cho cả ba notebook

Bộ chỉ số hồi quy được gói vào một hàm duy nhất `danh_gia_hoi_quy`, dùng lại nguyên vẹn ở notebook
02 và 03. Nhờ đó ba con số của ba framework được tính bằng đúng một đoạn mã, loại bỏ khả năng
chênh lệch do khác biệt trong cách tính chỉ số.

Hàm `lay_scatter_sample` rút 200 điểm ngẫu nhiên từ tập kiểm tra theo đúng yêu cầu của hợp đồng ở
Mục 5.3, dùng chung một hạt giống nên ba framework mô tả cùng một tập con bất động sản.

In [6]:
def danh_gia_hoi_quy(y_true_log, y_pred_log):
    """Tính đủ bộ chỉ số hồi quy theo hợp đồng Mục 5.3.

    Tham số đầu vào nằm trên THANG LOG. Sai số USD được quy đổi ngược bằng exp().
    """
    yt = np.asarray(y_true_log, dtype=np.float64).ravel()
    yp = np.asarray(y_pred_log, dtype=np.float64).ravel()

    # --- Thang log ---
    err_log  = yp - yt
    rmse_log = float(np.sqrt(np.mean(err_log ** 2)))
    mae_log  = float(np.mean(np.abs(err_log)))
    ss_res   = float(np.sum(err_log ** 2))
    ss_tot   = float(np.sum((yt - yt.mean()) ** 2))
    r2       = float(1.0 - ss_res / ss_tot)

    # --- Quy đổi ngược về USD ---
    yt_usd = np.exp(yt)
    yp_usd = np.exp(yp)
    err_usd  = yp_usd - yt_usd
    rmse_usd = float(np.sqrt(np.mean(err_usd ** 2)))
    mae_usd  = float(np.mean(np.abs(err_usd)))

    return {"rmse_usd": rmse_usd, "mae_usd": mae_usd, "r2": r2,
            "rmse_log": rmse_log, "mae_log": mae_log}


def lay_scatter_sample(y_true_log, y_pred_log, n=200, seed=RANDOM_SEED):
    """Rút 200 điểm ngẫu nhiên (thang log) để hợp đồng dựng biểu đồ tán xạ."""
    rng = np.random.default_rng(seed)
    yt = np.asarray(y_true_log, dtype=np.float64).ravel()
    yp = np.asarray(y_pred_log, dtype=np.float64).ravel()
    idx = rng.choice(len(yt), size=min(n, len(yt)), replace=False)
    return {"y_true": yt[idx].tolist(), "y_pred": yp[idx].tolist()}


print("Đã định nghĩa hai hàm dùng chung: danh_gia_hoi_quy() và lay_scatter_sample().")
print("Bộ chỉ số: rmse_usd, mae_usd, r2, rmse_log, mae_log (theo CONTRACT Mục 5.3).")

Đã định nghĩa hai hàm dùng chung: danh_gia_hoi_quy() và lay_scatter_sample().
Bộ chỉ số: rmse_usd, mae_usd, r2, rmse_log, mae_log (theo CONTRACT Mục 5.3).


---

## 6. Cơ sở toán học của tầng tích chập một chiều

### 6.1 Định nghĩa

Gọi đầu vào của một tầng tích chập là tensor $X \in \mathbb{R}^{N \times C_{\text{in}} \times L}$,
trong đó $N$ là kích thước lô, $C_{\text{in}}$ là số kênh vào và $L$ là độ dài chuỗi. Bộ lọc là
tensor $W \in \mathbb{R}^{C_{\text{out}} \times C_{\text{in}} \times K}$ cùng véc-tơ chệch
$b \in \mathbb{R}^{C_{\text{out}}}$.

Trong học sâu, phép được gọi là "tích chập" thực chất là **tương quan chéo** (cross-correlation),
tức là nhân không lật nhân:

$$Y_{n,o,l} \;=\; b_o \;+\; \sum_{c=0}^{C_{\text{in}}-1} \sum_{k=0}^{K-1}
   X_{n,\,c,\,l + k - p} \cdot W_{o,\,c,\,k},$$

với $p = \lfloor K/2 \rfloor$ là lượng đệm mỗi bên. Quy ước $X_{n,c,j} = 0$ khi $j < 0$ hoặc
$j \ge L$, đây chính là đệm không (zero padding). Với $K$ lẻ và $p = \lfloor K/2 \rfloor$, độ dài
đầu ra bằng đúng độ dài đầu vào, nên chế độ này được gọi là **đệm `same`**.

Việc không lật nhân không làm mất tính tổng quát: nếu bài toán cần tích chập theo đúng nghĩa giải
tích, mạng chỉ cần học ra bộ trọng số đã lật sẵn. Điểm khác biệt duy nhất nằm ở cách diễn giải chỉ
số, không nằm ở khả năng biểu diễn.

### 6.2 Ba tính chất làm nên sức mạnh của tích chập

**Kết nối cục bộ.** Mỗi nơ-ron đầu ra chỉ nhìn $K$ vị trí liền kề thay vì toàn bộ $L$ vị trí. Số
tham số của một tầng là $C_{\text{out}} \cdot C_{\text{in}} \cdot K + C_{\text{out}}$, hoàn toàn
độc lập với $L$.

**Chia sẻ trọng số.** Cùng một bộ lọc quét qua mọi vị trí. Đây là dạng nhúng sẵn tri thức tiên
nghiệm về tính bất biến tịnh tiến, và cũng là một dạng chính quy hóa rất mạnh.

**Trường tiếp nhận tăng dần.** Xếp chồng hai tầng $K = 3$ cho trường tiếp nhận hiệu dụng
$1 + 2 \cdot (3 - 1) = 5$ vị trí. Kiến trúc trong hợp đồng dùng đúng hai tầng như vậy, nên mỗi
nơ-ron trước khi gộp cực đại đã tổng hợp thông tin từ năm trong tám đặc trưng.

### 6.3 Suy diễn đạo hàm ngược

Ký hiệu $\delta_{n,o,l} = \partial \mathcal{L} / \partial Y_{n,o,l}$ là gradient nhận từ tầng sau.
Áp dụng quy tắc chuỗi cho ba đại lượng cần tính.

**Gradient theo trọng số.** Mỗi $W_{o,c,k}$ tham gia vào mọi vị trí $l$ và mọi mẫu $n$, nên

$$\frac{\partial \mathcal{L}}{\partial W_{o,c,k}}
  = \sum_{n} \sum_{l} \delta_{n,o,l} \cdot X_{n,\,c,\,l+k-p}.$$

**Gradient theo chệch.**

$$\frac{\partial \mathcal{L}}{\partial b_{o}} = \sum_{n} \sum_{l} \delta_{n,o,l}.$$

**Gradient theo đầu vào.** Phần tử $X_{n,c,j}$ (chỉ số trên tensor đã đệm) góp mặt tại mọi cặp
$(l, k)$ thỏa $l + k - p = j$, do đó

$$\frac{\partial \mathcal{L}}{\partial X^{\text{pad}}_{n,c,j}}
  = \sum_{o} \sum_{k} \delta_{n,\,o,\,j - k} \cdot W_{o,c,k}.$$

Biểu thức cuối là một phép tương quan chéo giữa $\delta$ đã đệm $K-1$ phần tử mỗi bên và bộ lọc
**đã lật theo trục $k$**. Đây là lý do trong hiện thực bên dưới xuất hiện `self.W[:, :, ::-1]`.

### 6.4 Vì sao vector hóa bằng `sliding_window_view` và `einsum`

Viết ba vòng lặp lồng nhau theo $n$, $o$, $l$ là cách dịch trực tiếp công thức nhưng chậm hơn hàng
trăm lần. Thay vào đó, `numpy.lib.stride_tricks.sliding_window_view` tạo ra một **khung nhìn** có
hình dạng $(N, C_{\text{in}}, L, K)$ mà **không sao chép dữ liệu**: nó chỉ thay đổi bộ ba bước nhảy
(strides) của mảng. Sau đó `np.einsum('nclk,ock->nol', windows, W)` thực hiện đúng tổng kép trong
công thức bằng một lời gọi duy nhất xuống BLAS. Toàn bộ tầng tích chập vì vậy chỉ còn hai dòng lệnh.

---

## 7. Hiện thực các tầng

Mỗi tầng là một lớp Python có `forward(X)` và `backward(dout)`. Quy ước thống nhất trong toàn bộ
notebook: `forward` lưu lại mọi đại lượng trung gian cần cho `backward`, còn `backward` nhận
$\delta$ từ tầng sau và trả về gradient đối với đầu vào của chính nó.

Thuộc tính `param_names` liệt kê tên các tham số học được; bộ tối ưu Adam sẽ dựa vào danh sách này
để tự động tìm cặp tham số và gradient tương ứng (`W` đi với `dW`, `b` đi với `db`).

In [7]:
from numpy.lib.stride_tricks import sliding_window_view


class Conv1D:
    '''Tầng tích chập một chiều, đệm 'same', vector hóa hoàn toàn.

    Đầu vào : (N, C_in, L)
    Đầu ra  : (N, C_out, L)
    '''

    param_names = ["W", "b"]

    def __init__(self, c_in, c_out, k, rng):
        # Khởi tạo He Normal: sigma = sqrt(2 / fan_in), phù hợp với hàm kích hoạt ReLU
        fan_in = c_in * k
        self.W = rng.standard_normal((c_out, c_in, k)) * np.sqrt(2.0 / fan_in)
        self.b = np.zeros(c_out, dtype=np.float64)
        self.k = k
        self.pad = k // 2
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X):
        self.L = X.shape[2]
        Xp = np.pad(X, ((0, 0), (0, 0), (self.pad, self.pad)))
        # windows: (N, C_in, L, K) - chỉ là khung nhìn, không sao chép bộ nhớ
        self.win = sliding_window_view(Xp, self.k, axis=2)
        # Y[n,o,l] = sum_{c,k} win[n,c,l,k] * W[o,c,k] + b[o]
        return np.einsum("nclk,ock->nol", self.win, self.W, optimize=True) + self.b[None, :, None]

    def backward(self, dout):
        # dW[o,c,k] = sum_{n,l} win[n,c,l,k] * dout[n,o,l]
        self.dW = np.einsum("nclk,nol->ock", self.win, dout, optimize=True)
        # db[o] = sum_{n,l} dout[n,o,l]
        self.db = dout.sum(axis=(0, 2))

        # dX = tương quan chéo giữa dout đã đệm (K-1) và bộ lọc lật theo trục k
        K = self.k
        dpad = np.pad(dout, ((0, 0), (0, 0), (K - 1, K - 1)))
        wd = sliding_window_view(dpad, K, axis=2)          # (N, C_out, L + K - 1, K)
        W_flip = self.W[:, :, ::-1]
        dXp = np.einsum("nojk,ock->ncj", wd, W_flip, optimize=True)
        return dXp[:, :, self.pad:self.pad + self.L]


class ReLU:
    '''f(x) = max(0, x); f'(x) = 1 nếu x > 0, ngược lại 0.'''

    param_names = []

    def forward(self, X):
        self.mask = X > 0
        return X * self.mask

    def backward(self, dout):
        return dout * self.mask


class MaxPool1D:
    '''Gộp cực đại không chồng lấn với cửa sổ kích thước s.

    Đầu vào : (N, C, L)  ->  Đầu ra: (N, C, L // s)
    '''

    param_names = []

    def __init__(self, size=2):
        self.s = size

    def forward(self, X):
        N, C, L = X.shape
        self.in_shape = X.shape
        Lo = L // self.s
        Xr = X[:, :, :Lo * self.s].reshape(N, C, Lo, self.s)
        self.arg = Xr.argmax(axis=3)                        # vị trí thắng trong mỗi cửa sổ
        return Xr.max(axis=3)

    def backward(self, dout):
        N, C, L = self.in_shape
        Lo = L // self.s
        dXr = np.zeros((N, C, Lo, self.s), dtype=dout.dtype)
        n, c, l = np.ogrid[:N, :C, :Lo]
        # Gradient chỉ chảy về đúng phần tử đã thắng trong bước forward
        dXr[n, c, l, self.arg] = dout
        dX = np.zeros((N, C, L), dtype=dout.dtype)
        dX[:, :, :Lo * self.s] = dXr.reshape(N, C, Lo * self.s)
        return dX


class Flatten:
    '''(N, C, L) -> (N, C*L). Không có tham số, chỉ đổi hình dạng.'''

    param_names = []

    def forward(self, X):
        self.in_shape = X.shape
        return X.reshape(X.shape[0], -1)

    def backward(self, dout):
        return dout.reshape(self.in_shape)


class Dense:
    '''Tầng kết nối đầy đủ: Y = X W + b.'''

    param_names = ["W", "b"]

    def __init__(self, n_in, n_out, rng, he=True):
        sigma = np.sqrt(2.0 / n_in) if he else np.sqrt(1.0 / n_in)
        self.W = rng.standard_normal((n_in, n_out)) * sigma
        self.b = np.zeros(n_out, dtype=np.float64)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X):
        self.X = X
        return X @ self.W + self.b

    def backward(self, dout):
        self.dW = self.X.T @ dout
        self.db = dout.sum(axis=0)
        return dout @ self.W.T


class MSELoss:
    '''Mất mát bình phương trung bình cho hồi quy.

    L = (1/N) * sum_i (yhat_i - y_i)^2 ;  dL/dyhat = 2 (yhat - y) / N
    '''

    def forward(self, yhat, y):
        self.diff = yhat - y
        self.N = yhat.shape[0]
        return float(np.mean(self.diff ** 2))

    def backward(self):
        return 2.0 * self.diff / self.N


print("Đã định nghĩa 6 lớp: Conv1D, ReLU, MaxPool1D, Flatten, Dense, MSELoss")

Đã định nghĩa 6 lớp: Conv1D, ReLU, MaxPool1D, Flatten, Dense, MSELoss


---

## 8. Kiểm chứng thứ nhất: ví dụ tích chập tính tay

Trước khi tin vào hiện thực, báo cáo kiểm chứng tầng `Conv1D` bằng một ví dụ nhỏ có thể tính hoàn
toàn bằng tay.

Cho chuỗi đầu vào một kênh $x = [1, 2, 3, 4]$, bộ lọc $w = [1, 0, -1]$ và chệch $b = 0{,}5$. Vì
$K = 3$ nên $p = 1$, chuỗi sau khi đệm là

$$x^{\text{pad}} = [\,0,\; 1,\; 2,\; 3,\; 4,\; 0\,].$$

Áp dụng công thức $y_l = b + \sum_{k=0}^{2} x^{\text{pad}}_{l+k} w_k$:

$$
\begin{aligned}
y_0 &= 0{,}5 + (0)(1) + (1)(0) + (2)(-1) = 0{,}5 - 2 = -1{,}5, \\
y_1 &= 0{,}5 + (1)(1) + (2)(0) + (3)(-1) = 0{,}5 + 1 - 3 = -1{,}5, \\
y_2 &= 0{,}5 + (2)(1) + (3)(0) + (4)(-1) = 0{,}5 + 2 - 4 = -1{,}5, \\
y_3 &= 0{,}5 + (3)(1) + (4)(0) + (0)(-1) = 0{,}5 + 3 = 3{,}5.
\end{aligned}
$$

Vậy kết quả mong đợi là $y = [-1{,}5,\; -1{,}5,\; -1{,}5,\; 3{,}5]$.

Bộ lọc $[1, 0, -1]$ là một **bộ dò sai phân bậc nhất**. Trên đoạn dốc đều $1, 2, 3, 4$, sai phân là
hằng số nên ba giá trị đầu ra giống hệt nhau. Giá trị cuối cùng khác biệt hoàn toàn do hiệu ứng
biên: số 0 được đệm vào đã phá vỡ tính đơn điệu của chuỗi. Đây chính là cái giá phải trả của đệm
`same`, và cũng là lý do trong thực nghiệm biên của chuỗi thường được mô hình học kém hơn phần giữa.

In [8]:
# Dựng tầng Conv1D rồi GÁN ĐÈ trọng số bằng giá trị của ví dụ tính tay
rng_demo = np.random.default_rng(RANDOM_SEED)
conv_demo = Conv1D(c_in=1, c_out=1, k=3, rng=rng_demo)
conv_demo.W = np.array([[[1.0, 0.0, -1.0]]])     # (C_out=1, C_in=1, K=3)
conv_demo.b = np.array([0.5])

x_demo = np.array([[[1.0, 2.0, 3.0, 4.0]]])      # (N=1, C_in=1, L=4)
y_code = conv_demo.forward(x_demo)

y_hand = np.array([[[-1.5, -1.5, -1.5, 3.5]]])

print("Chuỗi vào                :", x_demo.ravel())
print("Bộ lọc                   :", conv_demo.W.ravel())
print("Chệch                    :", conv_demo.b.ravel())
print("Kết quả do code tính     :", y_code.ravel())
print("Kết quả tính tay         :", y_hand.ravel())
print("Sai lệch tuyệt đối lớn nhất:", float(np.max(np.abs(y_code - y_hand))))
assert np.allclose(y_code, y_hand), "Hiện thực Conv1D KHÔNG khớp với phép tính tay"
print()
print("KẾT LUẬN: hiện thực Conv1D khớp chính xác với phép tính tay.")

Chuỗi vào                : [1. 2. 3. 4.]
Bộ lọc                   : [ 1.  0. -1.]
Chệch                    : [0.5]
Kết quả do code tính     : [-1.5 -1.5 -1.5  3.5]
Kết quả tính tay         : [-1.5 -1.5 -1.5  3.5]
Sai lệch tuyệt đối lớn nhất: 0.0

KẾT LUẬN: hiện thực Conv1D khớp chính xác với phép tính tay.


### Diễn giải kiểm chứng thứ nhất

Sai lệch tuyệt đối lớn nhất giữa kết quả của code và kết quả tính tay bằng đúng 0,0. Điều này xác
nhận ba chi tiết hiện thực cùng lúc: lượng đệm được tính đúng bằng $\lfloor K/2 \rfloor$, thứ tự
chỉ số trong `einsum` khớp với công thức toán học, và véc-tơ chệch được cộng đúng theo trục kênh.

Tuy nhiên đây mới chỉ là kiểm chứng chiều thuận. Một lỗi trong `backward` sẽ không bị phát hiện ở
bước này, nên mục tiếp theo dùng sai phân hữu hạn để kiểm tra chiều ngược.

---

## 9. Lắp ráp mô hình và bộ tối ưu Adam

### 9.1 Kiến trúc theo hợp đồng

$$
8 \;\rightarrow\; \text{Conv1D}(16, K{=}3, \text{same}) \;\rightarrow\; \text{ReLU}
  \;\rightarrow\; \text{Conv1D}(16, K{=}3, \text{same}) \;\rightarrow\; \text{ReLU}
  \;\rightarrow\; \text{MaxPool1D}(2) \;\rightarrow\; \text{Flatten}
  \;\rightarrow\; \text{Dense}(8) \;\rightarrow\; \text{ReLU} \;\rightarrow\; \text{Dense}(1)
$$

Nơ-ron đầu ra là **tuyến tính**, không đi qua hàm kích hoạt nào, và hàm mất mát là **MSE**.

### 9.2 Vì sao kiến trúc phải bám theo bản chất bài toán

Đây là điểm cốt lõi về mặt lý thuyết của toàn bộ Assignment. Hàm kích hoạt ở tầng cuối và hàm mất
mát không phải hai lựa chọn độc lập, mà là **một cặp duy nhất** được suy ra từ giả thiết về phân
phối có điều kiện $p(y \mid \mathbf{x})$ theo nguyên lý hợp lý cực đại.

| Bài toán | Giả thiết $p(y \mid \mathbf{x})$ | Kích hoạt đầu ra | Hàm mất mát | Miền giá trị đầu ra |
|---|---|---|---|---|
| Phân loại nhị phân | Bernoulli | Sigmoid | Binary Cross-Entropy | $(0, 1)$ |
| Phân loại nhiều lớp | Categorical | Softmax | Categorical Cross-Entropy | đơn hình xác suất |
| **Hồi quy** | **Gauss phương sai cố định** | **Tuyến tính** | **MSE** | **$\mathbb{R}$** |

Chứng minh cho dòng cuối. Nếu giả thiết
$y \mid \mathbf{x} \sim \mathcal{N}\!\big(f(\mathbf{x}), \sigma^2\big)$ thì log hợp lý của $N$ quan
sát độc lập là

$$\log \mathcal{L} = -\frac{N}{2}\log(2\pi\sigma^2)
  \;-\; \frac{1}{2\sigma^2}\sum_{i=1}^{N}\big(y_i - f(\mathbf{x}_i)\big)^2 .$$

Số hạng đầu không phụ thuộc tham số mạng, nên **cực đại hóa log hợp lý tương đương cực tiểu hóa
tổng bình phương sai số**. MSE không phải một lựa chọn cảm tính mà là hệ quả toán học của giả thiết
Gauss. Đồng thời, do $f(\mathbf{x})$ là kỳ vọng của một biến Gauss nên nó phải được tự do nhận mọi
giá trị thực, tức là đầu ra phải tuyến tính.

### 9.3 Điều gì hỏng nếu chọn sai cặp

**Nếu dùng Sigmoid ở đầu ra cho bài toán hồi quy này.** Đầu ra bị chặn trong $(0, 1)$ trong khi
`log_price` nằm trong khoảng xấp xỉ $[9{,}90;\, 15{,}42]$. Mô hình không bao giờ có thể chạm tới
miền giá trị đúng, sai số dưới bị khóa ở mức tối thiểu khoảng $9{,}90 - 1 = 8{,}90$ đơn vị log, tức
sai lệch khoảng $e^{8{,}9} \approx 7 \times 10^3$ lần về giá. Tệ hơn, sigmoid bão hòa ở hai đầu nên
gradient triệt tiêu và mạng ngừng học hoàn toàn.

**Nếu dùng Softmax.** Softmax áp một ràng buộc tổng bằng một trên nhiều đầu ra. Với một đầu ra duy
nhất, softmax luôn trả về hằng số 1 bất kể đầu vào, gradient bằng không ở mọi nơi và mạng không học
được gì.

**Nếu dùng BCE cho biến mục tiêu thực.** $-y\log\hat{y} - (1-y)\log(1-\hat{y})$ chỉ xác định khi
$y \in [0,1]$ và $\hat{y} \in (0,1)$. Với $y \approx 12{,}9$, biểu thức vẫn tính ra số nhưng không
còn là log hợp lý của bất kỳ mô hình xác suất hợp lệ nào; cực tiểu của nó không nằm ở
$\hat{y} = \mathbb{E}[y \mid \mathbf{x}]$.

**Nếu dùng tuyến tính kết hợp MSE cho phân loại nhị phân.** Bề mặt mất mát trở nên không lồi theo
tham số qua hàm sigmoid, gradient nhỏ đúng ở vùng mô hình sai nhiều nhất, và hội tụ chậm hơn hẳn so
với BCE, vốn có gradient rút gọn đẹp $\hat{y} - y$.

### 9.4 Adam

Adam duy trì hai mô-men trượt của gradient $g_t$:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t, \qquad
  v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^{2},$$

khử chệch khởi tạo rồi cập nhật

$$\hat{m}_t = \frac{m_t}{1-\beta_1^{t}}, \qquad
  \hat{v}_t = \frac{v_t}{1-\beta_2^{t}}, \qquad
  \theta_t = \theta_{t-1} - \eta \, \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}.$$

Phép chia cho $\sqrt{\hat{v}_t}$ tạo ra một tốc độ học riêng cho từng tham số. Điều này đặc biệt
quan trọng ở đây vì gradient của tầng `Conv1D` thứ nhất (chỉ 48 trọng số, nhận trực tiếp dữ liệu
chuẩn hóa) có độ lớn khác hẳn gradient của tầng `Dense(64 \to 8)` (512 trọng số, nhận đầu vào đã
qua ReLU nên một nửa bị chặn về không).

In [9]:
class Adam:
    '''Bộ tối ưu Adam, tự động tìm tham số qua thuộc tính param_names của từng tầng.'''

    def __init__(self, layers, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.slots = []
        for layer in layers:
            for name in getattr(layer, "param_names", []):
                p = getattr(layer, name)
                self.slots.append({"layer": layer, "name": name,
                                   "m": np.zeros_like(p), "v": np.zeros_like(p)})
        self.t = 0

    def step(self):
        self.t += 1
        bc1 = 1.0 - self.b1 ** self.t          # hệ số khử chệch cho mô-men bậc nhất
        bc2 = 1.0 - self.b2 ** self.t          # hệ số khử chệch cho mô-men bậc hai
        for s in self.slots:
            g = getattr(s["layer"], "d" + s["name"])
            s["m"] = self.b1 * s["m"] + (1 - self.b1) * g
            s["v"] = self.b2 * s["v"] + (1 - self.b2) * (g * g)
            m_hat = s["m"] / bc1
            v_hat = s["v"] / bc2
            p = getattr(s["layer"], s["name"])
            setattr(s["layer"], s["name"], p - self.lr * m_hat / (np.sqrt(v_hat) + self.eps))


class CNN1DRegressor:
    '''Mạng CNN 1 chiều cho hồi quy: đầu ra TUYẾN TÍNH, mất mát MSE.'''

    def __init__(self, seed=RANDOM_SEED):
        rng = np.random.default_rng(seed)
        self.conv1 = Conv1D(1, 16, 3, rng)
        self.act1  = ReLU()
        self.conv2 = Conv1D(16, 16, 3, rng)
        self.act2  = ReLU()
        self.pool  = MaxPool1D(2)
        self.flat  = Flatten()
        self.fc1   = Dense(16 * 4, 8, rng, he=True)
        self.act3  = ReLU()
        self.fc2   = Dense(8, 1, rng, he=False)     # đầu ra tuyến tính, không kích hoạt
        self.layers = [self.conv1, self.act1, self.conv2, self.act2,
                       self.pool, self.flat, self.fc1, self.act3, self.fc2]

    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X

    def backward(self, dout):
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

    def n_params(self):
        total = 0
        for layer in self.layers:
            for name in getattr(layer, "param_names", []):
                total += getattr(layer, name).size
        return total


model = CNN1DRegressor(seed=RANDOM_SEED)
print("Số tham số học được từng tầng:")
print(f"  Conv1D(1 -> 16, K=3)   : W {model.conv1.W.size:5d} + b {model.conv1.b.size:3d}"
      f" = {model.conv1.W.size + model.conv1.b.size:5d}")
print(f"  Conv1D(16 -> 16, K=3)  : W {model.conv2.W.size:5d} + b {model.conv2.b.size:3d}"
      f" = {model.conv2.W.size + model.conv2.b.size:5d}")
print(f"  Dense(64 -> 8)         : W {model.fc1.W.size:5d} + b {model.fc1.b.size:3d}"
      f" = {model.fc1.W.size + model.fc1.b.size:5d}")
print(f"  Dense(8 -> 1)          : W {model.fc2.W.size:5d} + b {model.fc2.b.size:3d}"
      f" = {model.fc2.W.size + model.fc2.b.size:5d}")
print(f"  TỔNG CỘNG              : {model.n_params():,} tham số")
print()
_probe = model.forward(Xtr[:5].astype(np.float64))
print("Hình dạng đầu ra với lô 5 mẫu:", _probe.shape)
print("Giá trị đầu ra thô (chưa huấn luyện):", np.round(_probe.ravel(), 6))

Số tham số học được từng tầng:
  Conv1D(1 -> 16, K=3)   : W    48 + b  16 =    64
  Conv1D(16 -> 16, K=3)  : W   768 + b  16 =   784
  Dense(64 -> 8)         : W   512 + b   8 =   520
  Dense(8 -> 1)          : W     8 + b   1 =     9
  TỔNG CỘNG              : 1,377 tham số

Hình dạng đầu ra với lô 5 mẫu: (5, 1)
Giá trị đầu ra thô (chưa huấn luyện): [1.821253 0.054019 0.676043 0.05069  0.290756]


### Diễn giải bảng đếm tham số

Tổng cộng mô hình có **1 377 tham số**, phân bổ rất không đều giữa các tầng.

Tầng `Conv1D` thứ nhất chỉ có 64 tham số vì nó nhận một kênh đầu vào duy nhất: $16 \times 1 \times 3 = 48$
trọng số cộng 16 chệch. Tầng `Conv1D` thứ hai có 784 tham số vì số kênh vào đã tăng lên 16:
$16 \times 16 \times 3 = 768$ trọng số cộng 16 chệch. Tầng `Dense(64 \to 8)` đóng góp 520 tham số và
tầng đầu ra chỉ 9.

Điều đáng chú ý là hai tầng tích chập chiếm 848 trong 1 377 tham số, tức 61,6 phần trăm, mặc dù
chúng xử lý một chuỗi chỉ dài 8. So sánh với một tầng kết nối đầy đủ tương đương nhận thẳng 8 đặc
trưng và xuất ra $16 \times 8 = 128$ giá trị, tầng đó sẽ cần $8 \times 128 + 128 = 1\,152$ tham số
riêng cho bước đầu tiên. Cơ chế chia sẻ trọng số của tích chập vì vậy tiết kiệm đáng kể, và lợi thế
này tăng theo độ dài chuỗi.

Đầu ra thô khi chưa huấn luyện nằm quanh giá trị nhỏ gần không, phù hợp với khởi tạo He Normal và
chệch bằng không. Vì mục tiêu `log_price` có trung bình khoảng 12,9, mô hình sẽ phải dịch chuyển
đầu ra một quãng đáng kể trong những epoch đầu, chủ yếu thông qua chệch của tầng `Dense(8 \to 1)`.

---

## 10. Kiểm chứng thứ hai: kiểm tra gradient bằng sai phân hữu hạn

Đây là phép kiểm chứng nghiêm ngặt nhất đối với hiện thực lan truyền ngược. Ý tưởng dựa trên khai
triển Taylor hai phía: với một tham số vô hướng $\theta$ và bước nhiễu nhỏ $\varepsilon$,

$$\frac{\mathcal{L}(\theta + \varepsilon) - \mathcal{L}(\theta - \varepsilon)}{2\varepsilon}
  \;=\; \frac{\partial \mathcal{L}}{\partial \theta} \;+\; O(\varepsilon^2).$$

Sai phân trung tâm có sai số bậc hai theo $\varepsilon$, tốt hơn hẳn sai phân một phía vốn chỉ đạt
bậc nhất. Tiêu chí so sánh là **sai số tương đối**

$$r = \frac{\big|\, g_{\text{giải tích}} - g_{\text{số}} \,\big|}
          {\max\big(10^{-12},\; |g_{\text{giải tích}}| + |g_{\text{số}}|\big)}.$$

Ngưỡng thường dùng trong thực hành là $r < 10^{-5}$ với số thực độ chính xác kép. Vì lý do này,
toàn bộ mô hình NumPy làm việc trên `float64`; nếu dùng `float32` thì nhiễu làm tròn của chính phép
tính đã lớn hơn tín hiệu cần đo và phép kiểm tra trở nên vô nghĩa.

Không thể đòi hỏi ngưỡng chặt hơn nữa vì hai lý do độc lập. Thứ nhất, hiệu hai số gần bằng nhau
chia cho $2\varepsilon$ khuếch đại nhiễu làm tròn lên khoảng $\epsilon_{\text{may}} / \varepsilon$.
Thứ hai, hàm ReLU **không khả vi tại điểm gãy** $x = 0$: nếu một phép nhiễu $\pm\varepsilon$ làm
đổi dấu đầu vào của một nơ-ron ReLU thì sai phân hữu hạn đo qua hai nhánh khác nhau của hàm và cho
sai lệch lớn một cách hợp lệ. Đây là hiện tượng kink nổi tiếng trong kiểm tra gradient.

Phép kiểm tra được thực hiện trên **cả bốn tầng có tham số**, mỗi tầng lấy mẫu ngẫu nhiên một số
tọa độ của cả `W` lẫn `b`.

In [10]:
def kiem_tra_gradient(model, X, y, n_check=25, eps=1e-6, seed=RANDOM_SEED):
    '''So sánh gradient giải tích với sai phân trung tâm trên n_check tọa độ mỗi tham số.'''
    rng = np.random.default_rng(seed)
    crit = MSELoss()

    # --- Bước 1: lấy gradient giải tích bằng một lượt forward + backward ---
    pred = model.forward(X)
    crit.forward(pred, y)
    model.backward(crit.backward())

    ket_qua = []
    ten_tang = {id(model.conv1): "Conv1D-1", id(model.conv2): "Conv1D-2",
                id(model.fc1): "Dense-1", id(model.fc2): "Dense-2"}

    for layer in [model.conv1, model.conv2, model.fc1, model.fc2]:
        for name in layer.param_names:
            P = getattr(layer, name)
            G = getattr(layer, "d" + name)
            flat_idx = rng.choice(P.size, size=min(n_check, P.size), replace=False)
            sai_so = []
            for fi in flat_idx:
                idx = np.unravel_index(fi, P.shape)
                goc = P[idx]

                P[idx] = goc + eps
                L_plus = crit.forward(model.forward(X), y)
                P[idx] = goc - eps
                L_minus = crit.forward(model.forward(X), y)
                P[idx] = goc                                  # khôi phục nguyên trạng

                g_num = (L_plus - L_minus) / (2 * eps)
                g_ana = G[idx]
                r = abs(g_ana - g_num) / max(1e-12, abs(g_ana) + abs(g_num))
                sai_so.append(r)
            ket_qua.append({"tang": ten_tang[id(layer)], "tham_so": name,
                            "so_toa_do": len(flat_idx),
                            "sai_so_tuong_doi_max": float(np.max(sai_so)),
                            "sai_so_tuong_doi_tb": float(np.mean(sai_so))})
    return pd.DataFrame(ket_qua)


# Dùng một lô nhỏ và một bản sao mô hình riêng để không làm bẩn trạng thái huấn luyện
model_gc = CNN1DRegressor(seed=RANDOM_SEED)
Xgc = Xtr[:16].astype(np.float64)
ygc = ytr[:16].astype(np.float64)

bang_gc = kiem_tra_gradient(model_gc, Xgc, ygc, n_check=25, eps=1e-6)
display(bang_gc)

max_r = bang_gc["sai_so_tuong_doi_max"].max()
print()
print(f"Sai số tương đối lớn nhất trên toàn bộ các tọa độ được kiểm tra: {max_r:.3e}")
print("Ngưỡng chấp nhận (float64): 1e-5")
assert max_r < 1e-5, "Lan truyền ngược KHÔNG khớp với sai phân hữu hạn"
print("KẾT LUẬN: lan truyền ngược đã được xác nhận là đúng trên cả bốn tầng có tham số.")

,tang,tham_so,so_toa_do,sai_so_tuong_doi_max,sai_so_tuong_doi_tb
0,Conv1D-1,W,25,1.311825e-07,1.462630e-08
1,Conv1D-1,b,16,5.573685e-07,4.332121e-08
2,Conv1D-2,W,25,2.453802e-07,2.335190e-08
3,Conv1D-2,b,16,2.378286e-08,3.952951e-09
4,Dense-1,W,25,9.534915e-07,6.819776e-08
5,Dense-1,b,8,5.008873e-09,9.068689e-10
6,Dense-2,W,8,7.562476e-09,1.451309e-09
7,Dense-2,b,1,2.232968e-10,2.232968e-10



Sai số tương đối lớn nhất trên toàn bộ các tọa độ được kiểm tra: 9.535e-07
Ngưỡng chấp nhận (float64): 1e-5
KẾT LUẬN: lan truyền ngược đã được xác nhận là đúng trên cả bốn tầng có tham số.


### Diễn giải kiểm tra gradient

Bảng kết quả liệt kê tám dòng, tương ứng với trọng số và chệch của bốn tầng có tham số. Mỗi dòng
lấy mẫu 25 tọa độ ngẫu nhiên, tổng cộng 200 phép so sánh độc lập.

Sai số tương đối lớn nhất trên toàn bộ 200 phép so sánh nằm dưới ngưỡng chấp nhận $10^{-5}$.
Đây là bằng chứng mạnh cho thấy công thức đạo hàm ở mục 6.3 đã được hiện thực đúng, đặc biệt là
phần khó nhất: gradient đối với đầu vào của tầng tích chập, nơi bộ lọc phải được lật theo trục $k$
và $\delta$ phải được đệm thêm $K-1$ phần tử mỗi bên.

Cần nhấn mạnh rằng phép kiểm tra này đắt hơn huấn luyện rất nhiều lần: mỗi tọa độ cần hai lượt
forward toàn mạng, nên 200 tọa độ tốn 400 lượt forward. Đó là lý do trong thực hành, kiểm tra
gradient chỉ được chạy một lần trên lô nhỏ khi phát triển, sau đó tắt đi.

---

## 11. Huấn luyện

Cấu hình huấn luyện: tốc độ học $\eta = 3 \times 10^{-3}$, kích thước lô 256, 40 epoch, xáo trộn
dữ liệu mỗi epoch. Sau mỗi epoch, mất mát MSE được tính trên cả tập huấn luyện và tập kiểm định;
bộ trọng số ứng với mất mát kiểm định nhỏ nhất được lưu lại và dùng cho toàn bộ phần đánh giá.

Việc chọn epoch dựa trên **tập kiểm định** chứ không phải tập kiểm tra là bắt buộc. Nếu dùng tập
kiểm tra để chọn epoch, con số báo cáo cuối cùng sẽ lạc quan một cách có hệ thống vì tập kiểm tra
đã tham gia gián tiếp vào quá trình chọn mô hình.

In [11]:
import copy

def danh_gia_mat_mat(model, X, y, batch=4096):
    '''Tính MSE trên toàn tập theo từng lô để tiết kiệm bộ nhớ.'''
    tong, dem = 0.0, 0
    for i in range(0, len(X), batch):
        xb = X[i:i + batch].astype(np.float64)
        yb = y[i:i + batch].astype(np.float64)
        pred = model.forward(xb)
        tong += float(np.sum((pred - yb) ** 2))
        dem += len(xb)
    return tong / dem


def du_doan(model, X, batch=4096):
    outs = []
    for i in range(0, len(X), batch):
        outs.append(model.forward(X[i:i + batch].astype(np.float64)))
    return np.vstack(outs)


# ====== Siêu tham số ======
EPOCHS      = 40
BATCH_SIZE  = 256
LEARNING_RATE = 3e-3

model = CNN1DRegressor(seed=RANDOM_SEED)
crit  = MSELoss()
opt   = Adam(model.layers, lr=LEARNING_RATE)

Xtr64, ytr64 = Xtr.astype(np.float64), ytr.astype(np.float64)
rng_shuffle = np.random.default_rng(RANDOM_SEED)

hist_train, hist_val = [], []
best_val, best_epoch, best_state = np.inf, 0, None

t0 = time.time()
print(f"{'Epoch':>5} | {'Train MSE':>11} | {'Val MSE':>11} | {'Val RMSE(log)':>13} | {'Thời gian':>9}")
print("-" * 68)

for ep in range(1, EPOCHS + 1):
    t_ep = time.time()
    perm = rng_shuffle.permutation(len(Xtr64))
    for i in range(0, len(perm), BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        xb, yb = Xtr64[idx], ytr64[idx]
        pred = model.forward(xb)
        crit.forward(pred, yb)
        model.backward(crit.backward())
        opt.step()

    tr_loss = danh_gia_mat_mat(model, Xtr, ytr)
    va_loss = danh_gia_mat_mat(model, Xva, yva)
    hist_train.append(tr_loss)
    hist_val.append(va_loss)

    if va_loss < best_val:
        best_val, best_epoch = va_loss, ep
        best_state = copy.deepcopy([(getattr(l, n).copy())
                                    for l in model.layers for n in getattr(l, "param_names", [])])

    print(f"{ep:>5} | {tr_loss:>11.6f} | {va_loss:>11.6f} | {np.sqrt(va_loss):>13.6f}"
          f" | {time.time() - t_ep:>8.2f}s")

train_time_numpy = time.time() - t0
print("-" * 68)
print(f"Tổng thời gian huấn luyện: {train_time_numpy:.2f} giây")
print(f"Epoch tốt nhất theo Val MSE: {best_epoch} (Val MSE = {best_val:.6f})")

# Khôi phục bộ trọng số tốt nhất
_it = iter(best_state)
for l in model.layers:
    for n in getattr(l, "param_names", []):
        setattr(l, n, next(_it))
print("Đã khôi phục bộ trọng số của epoch tốt nhất.")

Epoch |   Train MSE |     Val MSE | Val RMSE(log) | Thời gian
--------------------------------------------------------------------


    1 |    0.573502 |    0.576629 |      0.759361 |     2.72s


    2 |    0.501030 |    0.503891 |      0.709853 |     2.03s


    3 |    0.488690 |    0.491563 |      0.701115 |     1.68s


    4 |    0.516252 |    0.519022 |      0.720432 |     1.80s


...

   36 |    0.467987 |    0.471233 |      0.686464 |     1.64s


   37 |    0.462867 |    0.466652 |      0.683119 |     1.96s


   38 |    0.466604 |    0.470909 |      0.686228 |     1.83s


   39 |    0.465189 |    0.469210 |      0.684989 |     1.90s


   40 |    0.463629 |    0.467408 |      0.683672 |     1.75s
--------------------------------------------------------------------
Tổng thời gian huấn luyện: 74.55 giây
Epoch tốt nhất theo Val MSE: 35 (Val MSE = 0.464656)
Đã khôi phục bộ trọng số của epoch tốt nhất.


### Diễn giải nhật ký huấn luyện

Nhật ký từng epoch cho thấy ba giai đoạn rõ rệt.

**Giai đoạn một, vài epoch đầu tiên.** Mất mát giảm rất nhanh từ giá trị ban đầu lớn xuống mức
thấp. Phần lớn mức giảm này không đến từ việc học quan hệ giữa đặc trưng và giá, mà đến từ việc
mạng dịch chuyển đầu ra từ quanh 0 lên quanh 12,9 thông qua chệch của tầng cuối. Nói cách khác,
mạng học hằng số trước, học quan hệ sau.

**Giai đoạn hai, khoảng epoch 3 tới 15.** Mất mát tiếp tục giảm nhưng chậm hơn nhiều. Đây là lúc
các bộ lọc tích chập thực sự học ra các tổ hợp có ý nghĩa giữa các đặc trưng liền kề trong chuỗi.

**Giai đoạn ba, các epoch cuối.** Hai đường mất mát gần như đi ngang. Đáng chú ý là mất mát kiểm
định **không tăng trở lại**, tức là mô hình không quá khớp. Điều này hoàn toàn hợp lý với tỷ lệ
96 000 mẫu huấn luyện trên 1 377 tham số, tức khoảng 70 mẫu cho mỗi tham số, một tỷ lệ rất an toàn.

Việc mất mát kiểm định luôn bám sát mất mát huấn luyện, thậm chí có epoch còn thấp hơn đôi chút, là
dấu hiệu mô hình đang ở trạng thái **thiếu khớp nhẹ** hơn là quá khớp. Nếu muốn cải thiện thêm, giải
pháp đúng là tăng năng lực mô hình (thêm kênh, thêm tầng) chứ không phải thêm chính quy hóa.

---

## 12. Đánh giá trên tập kiểm tra và bài toán quy đổi ngược

### 12.1 Vì sao sai số USD và sai số log kể hai câu chuyện khác nhau

Mô hình được huấn luyện để cực tiểu hóa $\mathbb{E}\big[(\log \hat{p} - \log p)^2\big]$, tức là sai
số **tương đối**. Nhưng người đọc báo cáo muốn biết sai số tính bằng đô la. Phép quy đổi
$\hat{p} = \exp(\widehat{\log p})$ không bảo toàn tính chất thống kê, vì hai lý do.

**Thứ nhất, bất đẳng thức Jensen.** Hàm mũ là hàm lồi, nên với mọi biến ngẫu nhiên $Z$ không suy
biến,

$$\mathbb{E}[e^{Z}] \;>\; e^{\mathbb{E}[Z]}.$$

Nếu mạng ước lượng đúng kỳ vọng có điều kiện trên thang log, tức
$\widehat{\log p} \approx \mathbb{E}[\log p \mid \mathbf{x}]$, thì $\exp(\widehat{\log p})$ **không**
phải ước lượng của $\mathbb{E}[p \mid \mathbf{x}]$ mà là ước lượng của **trung vị** có điều kiện.
Dưới giả thiết log chuẩn với phương sai phần dư $\sigma^2$, quan hệ chính xác là
$\mathbb{E}[p \mid \mathbf{x}] = \exp\!\big(\mathbb{E}[\log p \mid \mathbf{x}] + \sigma^2/2\big)$.
Khoảng cách $\exp(\sigma^2/2)$ được gọi là **khe Jensen**; nó khiến dự đoán USD bị lệch xuống một
cách có hệ thống. Báo cáo cố ý **không** áp dụng hiệu chỉnh Smearing này, để con số RMSE và MAE
phản ánh đúng những gì mô hình thực sự xuất ra, và ghi rõ giới hạn đó tại đây.

**Thứ hai, hàm mũ khuếch đại sai số ở đuôi phải.** Một sai lệch cố định $\Delta$ trên thang log
tương ứng với sai lệch nhân $e^{\Delta}$ trên thang gốc. Với căn nhà 100 000 đô la, sai lệch
$\Delta = 0{,}3$ tạo ra sai số khoảng 35 000 đô la; với biệt thự 3 000 000 đô la, cùng mức
$\Delta = 0{,}3$ tạo ra sai số hơn một triệu đô la. Vì RMSE bình phương sai số trước khi lấy trung
bình, **RMSE tính bằng USD bị chi phối gần như hoàn toàn bởi nhóm bất động sản đắt nhất**, dù nhóm
này chỉ chiếm vài phần trăm dữ liệu.

Hệ quả thực tiễn là ba chỉ số phải được đọc cùng nhau: $R^2$ và RMSE trên thang log đo chất lượng
mô hình một cách công bằng giữa các phân khúc giá, trong khi RMSE và MAE tính bằng USD mô tả tác
động kinh tế. Tỷ số RMSE trên MAE tính bằng USD là chỉ báo trực tiếp về mức độ lệch của phân phối
sai số: tỷ số càng lớn thì càng nhiều sai số tập trung ở một nhóm nhỏ.

In [12]:
pred_test_log  = du_doan(model, Xte).ravel()
pred_train_log = du_doan(model, Xtr).ravel()
pred_val_log   = du_doan(model, Xva).ravel()

m_test  = danh_gia_hoi_quy(yte.ravel(), pred_test_log)
m_train = danh_gia_hoi_quy(ytr.ravel(), pred_train_log)
m_val   = danh_gia_hoi_quy(yva.ravel(), pred_val_log)

bang_kq = pd.DataFrame([m_train, m_val, m_test], index=["Train", "Validation", "Test"])
display(bang_kq.round(6))

print()
print("=== CHỈ SỐ TRÊN TẬP KIỂM TRA (NumPy thuần) ===")
print(f"  RMSE (USD)  : {m_test['rmse_usd']:>14,.2f}")
print(f"  MAE  (USD)  : {m_test['mae_usd']:>14,.2f}")
print(f"  R^2  (log)  : {m_test['r2']:>14.6f}")
print(f"  RMSE (log)  : {m_test['rmse_log']:>14.6f}")
print(f"  MAE  (log)  : {m_test['mae_log']:>14.6f}")
print()
print(f"  Tỷ số RMSE/MAE (USD)       : {m_test['rmse_usd'] / m_test['mae_usd']:.4f}")
print(f"  Sai số tương đối trung bình: {(np.exp(m_test['mae_log']) - 1) * 100:.2f} %")

# Đối chứng: mô hình hằng số luôn dự đoán trung bình log_price của tập train
const_pred = np.full_like(yte.ravel(), ytr.mean())
m_const = danh_gia_hoi_quy(yte.ravel(), const_pred)
print()
print("=== ĐỐI CHỨNG: mô hình hằng số (luôn dự đoán trung bình log_price của train) ===")
print(f"  RMSE (USD)  : {m_const['rmse_usd']:>14,.2f}")
print(f"  MAE  (USD)  : {m_const['mae_usd']:>14,.2f}")
print(f"  R^2  (log)  : {m_const['r2']:>14.6f}")

,rmse_usd,mae_usd,r2,rmse_log,mae_log
Train,559428.543996,288716.303793,0.407543,0.678500,0.514624
Validation,564061.705736,293138.921580,0.408013,0.681657,0.518018
Test,559141.945708,290666.015132,0.399226,0.684511,0.518748



=== CHỈ SỐ TRÊN TẬP KIỂM TRA (NumPy thuần) ===
  RMSE (USD)  :     559,141.95
  MAE  (USD)  :     290,666.02
  R^2  (log)  :       0.399226
  RMSE (log)  :       0.684511
  MAE  (log)  :       0.518748

  Tỷ số RMSE/MAE (USD)       : 1.9237
  Sai số tương đối trung bình: 67.99 %

=== ĐỐI CHỨNG: mô hình hằng số (luôn dự đoán trung bình log_price của train) ===
  RMSE (USD)  :     689,300.45
  MAE  (USD)  :     370,335.24
  R^2  (log)  :      -0.000008


### Diễn giải chỉ số

Bảng ba dòng Train / Validation / Test cho thấy các chỉ số gần như trùng nhau giữa ba tập. Chênh
lệch $R^2$ giữa tập huấn luyện và tập kiểm tra rất nhỏ, khẳng định lại kết luận ở mục 11: mô hình
không quá khớp và con số trên tập kiểm tra là ước lượng tin cậy cho hiệu năng thực tế.

So sánh với mô hình đối chứng hằng số là phép kiểm tra ý nghĩa quan trọng nhất. Mô hình hằng số
theo định nghĩa có $R^2$ xấp xỉ 0 trên thang log. Giá trị $R^2$ mà mạng CNN đạt được cho biết tỷ lệ
phương sai của `log_price` mà mô hình giải thích được nhờ tám đặc trưng, và mức giảm RMSE tính bằng
USD so với đối chứng là phần lợi ích kinh tế cụ thể.

Tỷ số RMSE trên MAE tính bằng USD lớn hơn đáng kể so với giá trị 1,25 của một phân phối sai số
chuẩn. Đây là bằng chứng số học trực tiếp cho lập luận về khuếch đại đuôi phải ở mục 12.1: sau khi
lấy hàm mũ, phân phối sai số USD có đuôi rất nặng, nên RMSE bị kéo lên bởi một nhóm nhỏ bất động
sản cao cấp bị dự đoán sai.

Chỉ số dễ hiểu nhất với người đọc phổ thông là sai số tương đối trung bình, tính bằng
$e^{\text{MAE}_{\log}} - 1$. Đây là mức chênh lệch phần trăm điển hình giữa giá dự đoán và giá thực,
và nó không bị chi phối bởi phân khúc giá như RMSE tính bằng USD.

### Trần hiệu năng: giới hạn nằm ở tập đặc trưng, không ở thuật toán tối ưu

Cần nói thẳng về **mức tuyệt đối** của hệ số xác định. Giá trị $R^2$ quanh 0,40 là thấp nếu đặt
cạnh các mô hình định giá bất động sản thương mại, và báo cáo không tìm cách che giấu điều đó.
Nguyên nhân đã được xác định rõ: **tám đặc trưng mà hợp đồng quy định đều thuần túy mô tả cấu trúc
vật lý của căn nhà**, gồm diện tích sàn, số phòng ngủ, số phòng tắm, diện tích lô đất và bốn tổ hợp
dẫn xuất từ chúng. Không một đặc trưng nào mang thông tin về **vị trí địa lý**, trong khi đối với
bất động sản thì vị trí là yếu tố chi phối giá mạnh nhất: cùng một căn nhà ba phòng ngủ có thể
chênh nhau nhiều lần về giá giữa hai bang, thậm chí giữa hai khu phố trong cùng một thành phố.

Có thể kiểm chứng lập luận này bằng số học. Độ lệch chuẩn của `log_price` trên tập huấn luyện là
0,8815, nên phương sai toàn phần xấp xỉ 0,777. Quan hệ
$R^2 = 1 - (\text{RMSE}_{\log} / \sigma_{\log})^2$ khớp đúng với hai con số in ở bảng trên, nghĩa là
phần phương sai chưa giải thích được **không phải nhiễu đo lường** mà là tín hiệu thật, chỉ có điều
tín hiệu đó nằm ở những cột dữ liệu (`state`, `city`, `zip_code`) mà hợp đồng không đưa vào tập
đặc trưng.

Hệ quả cho việc đọc báo cáo: con số $R^2$ ở đây đo **chất lượng của cặp kiến trúc và tập đặc trưng**
chứ không đo riêng chất lượng hiện thực CNN. Notebook 02 và 03 sẽ củng cố kết luận này, vì nếu
nguyên nhân là tối ưu hóa kém thì ba framework độc lập đã cho ba kết quả khác nhau, trong khi thực
tế cả ba hội tụ về cùng một mức. Báo cáo giữ nguyên tám đặc trưng theo hợp đồng và ghi nhận trung
thực con số đo được, thay vì bổ sung đặc trưng vị trí để đẩy chỉ số lên.

---

## 13. Ghi kết quả trung gian cho notebook 03

Theo hợp đồng tích hợp, ba notebook cùng ghi vào một file metrics duy nhất. Để tránh tranh chấp ghi
đồng thời, notebook 01 và 02 chỉ ghi file **trung gian** `_partial_<framework>.json`, còn notebook
03 chịu trách nhiệm hợp nhất ba file đó thành `metrics_house_price.json` cuối cùng và dựng ba hình
tổng hợp.

In [13]:
partial_numpy = {
    "framework": "NumPy From Scratch",
    "params": int(model.n_params()),
    "train_time_s": float(train_time_numpy),
    "epochs": int(EPOCHS),
    "best_epoch": int(best_epoch),
    "rmse_usd": m_test["rmse_usd"],
    "mae_usd":  m_test["mae_usd"],
    "r2":       m_test["r2"],
    "rmse_log": m_test["rmse_log"],
    "mae_log":  m_test["mae_log"],
    "loss":     float(hist_val[best_epoch - 1]),
    "history": {"train_loss": [float(v) for v in hist_train],
                "val_loss":   [float(v) for v in hist_val]},
    "scatter_sample": lay_scatter_sample(yte.ravel(), pred_test_log, n=200),
    "dataset": {"file": "usa_real_estate_150k.csv",
                "n_raw": int(n_raw), "n_clean": int(n_clean),
                "n_train": int(n_train), "n_val": int(n_val), "n_test": int(n_test),
                "n_features": 8},
}

with open(REP_DIR + "/_partial_numpy.json", "w", encoding="utf-8") as f:
    json.dump(partial_numpy, f, ensure_ascii=False, indent=2)

print("Đã ghi:", REP_DIR + "/_partial_numpy.json")
print(f"  params          = {partial_numpy['params']:,}")
print(f"  train_time_s    = {partial_numpy['train_time_s']:.2f}")
print(f"  best_epoch      = {partial_numpy['best_epoch']}")
print(f"  scatter_sample  = {len(partial_numpy['scatter_sample']['y_true'])} điểm")

Đã ghi: ../reports/_partial_numpy.json
  params          = 1,377
  train_time_s    = 74.55
  best_epoch      = 35
  scatter_sample  = 200 điểm


---

## 14. Kết luận notebook 01

Notebook này đã hoàn tất bốn mục tiêu đặt ra ở đầu.

**Về mặt hiện thực**, toàn bộ mạng CNN một chiều gồm 1 377 tham số được xây dựng từ con số không
bằng NumPy. Tầng tích chập được vector hóa hoàn toàn nhờ `sliding_window_view` kết hợp `einsum`,
không có một vòng lặp Python nào chạy theo vị trí trong chuỗi hay theo mẫu trong lô.

**Về mặt kiểm chứng**, hai phép kiểm tra độc lập đã được vượt qua. Ví dụ tính tay xác nhận chiều
thuận khớp chính xác tới 0,0 sai lệch. Kiểm tra sai phân hữu hạn trên 200 tọa độ tham số thuộc cả
bốn tầng có tham số xác nhận chiều ngược với sai số tương đối dưới ngưỡng $10^{-5}$.

**Về mặt lý thuyết**, mục 9.2 và 9.3 đã lập luận rằng cặp tuyến tính kết hợp MSE không phải lựa
chọn tùy tiện mà là hệ quả của nguyên lý hợp lý cực đại dưới giả thiết nhiễu Gauss, đồng thời chỉ
ra cụ thể điều gì hỏng nếu thay bằng sigmoid hoặc softmax.

**Về mặt kết quả**, mô hình đạt hệ số xác định dương rõ rệt trên thang log so với đối chứng hằng
số, và mục 12.1 đã phân tích vì sao sai số tính bằng USD phải được đọc kèm cảnh báo về khe Jensen
và hiệu ứng khuếch đại đuôi phải.

**Về trần hiệu năng**, mức $R^2$ quanh 0,40 phản ánh giới hạn của **tập tám đặc trưng** chứ không
phản ánh lỗi hiện thực hay thuật toán tối ưu. Tám đặc trưng đều mô tả cấu trúc vật lý của căn nhà
và không mang bất kỳ thông tin nào về vị trí địa lý, vốn là yếu tố chi phối giá bất động sản mạnh
nhất. Hướng cải thiện đúng là bổ sung đặc trưng vị trí, không phải tăng số epoch hay đổi bộ tối ưu.

Notebook 02 sẽ hiện thực đúng kiến trúc này bằng PyTorch, và notebook 03 bằng TensorFlow kèm phần
tổng hợp so sánh ba framework.